In [1]:
%%capture
!pip install textacy
!pip install faiss-gpu
!pip install chromadb
!pip install nltk
!pip install sentence_transformers

In [ ]:
%%capture
# Initiliase Ollama server on a thread

#Download ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread
#Download model
!ollama pull llama3
!pip install ollama
import ollama
model_name = 'llama3'

In [3]:
import re
import asyncio
import pandas as pd
import numpy as np
from pydantic import BaseModel
from ollama import AsyncClient
import textacy.datasets
import requests
import json
from sentence_transformers import SentenceTransformer
from textwrap import shorten
import nltk
from textacy.representations.vectorizers import Vectorizer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import torch


ModuleNotFoundError: No module named 'chromadb'

In [4]:
legal_topics = {
    "0": "None",
    "1": "involuntary confession",
    "2": "habeas corpus",
    "3": "plea bargaining: the constitutionality of and/or the circumstances of its exercise",
    "4": "retroactivity (of newly announced or newly enacted constitutional or statutory rights)",
    "5": "search and seizure (other than as pertains to vehicles or Crime Control Act)",
    "6": "search and seizure, vehicles",
    "7": "search and seizure, Crime Control Act",
    "8": "contempt of court or congress",
    "9": "self-incrimination (other than as pertains to Miranda or immunity from prosecution)",
    "10": "Miranda warnings",
    "11": "self-incrimination, immunity from prosecution",
    "12": "right to counsel (cf. indigents appointment of counsel or inadequate representation)",
    "13": "cruel and unusual punishment, death penalty (cf. extra legal jury influence, death penalty)",
    "14": "cruel and unusual punishment, non-death penalty (cf. liability, civil rights acts)",
    "15": "line-up",
    "16": "discovery and inspection (in the context of criminal litigation only, otherwise Freedom of Information Act and related federal or state statutes or regulations)",
    "17": "double jeopardy",
    "18": "ex post facto (state)",
    "19": "extra-legal jury influences: miscellaneous",
    "20": "extra-legal jury influences: prejudicial statements or evidence",
    "21": "extra-legal jury influences: contact with jurors outside courtroom",
    "22": "extra-legal jury influences: jury instructions (not necessarily in criminal cases)",
    "23": "extra-legal jury influences: voir dire (not necessarily a criminal case)",
    "24": "extra-legal jury influences: prison garb or appearance",
    "25": "extra-legal jury influences: jurors and death penalty (cf. cruel and unusual punishment)",
    "26": "extra-legal jury influences: pretrial publicity",
    "27": "confrontation (right to confront accuser, call and cross-examine witnesses)",
    "28": "subconstitutional fair procedure: confession of error",
    "29": "subconstitutional fair procedure: conspiracy (cf. Federal Rules of Criminal Procedure: conspiracy)",
    "30": "subconstitutional fair procedure: entrapment",
    "31": "subconstitutional fair procedure: exhaustion of remedies",
    "32": "subconstitutional fair procedure: fugitive from justice",
    "33": "subconstitutional fair procedure: presentation, admissibility, or sufficiency of evidence (not necessarily a criminal case)",
    "34": "subconstitutional fair procedure: stay of execution",
    "35": "subconstitutional fair procedure: timeliness",
    "36": "subconstitutional fair procedure: miscellaneous",
    "37": "Federal Rules of Criminal Procedure",
    "38": "statutory construction of criminal laws: assault",
    "39": "statutory construction of criminal laws: bank robbery",
    "40": "statutory construction of criminal laws: conspiracy (cf. subconstitutional fair procedure: conspiracy)",
    "41": "statutory construction of criminal laws: escape from custody",
    "42": "statutory construction of criminal laws: false statements (cf. statutory construction of criminal laws: perjury)",
    "43": "statutory construction of criminal laws: financial (other than in fraud or internal revenue)",
    "44": "statutory construction of criminal laws: firearms",
    "45": "statutory construction of criminal laws: fraud",
    "46": "statutory construction of criminal laws: gambling",
    "47": "statutory construction of criminal laws: Hobbs Act; i.e., 18 USC 1951",
    "48": "statutory construction of criminal laws: immigration (cf. immigration and naturalization)",
    "49": "statutory construction of criminal laws: internal revenue (cf. Federal Taxation)",
    "50": "statutory construction of criminal laws: Mann Act and related statutes",
    "51": "statutory construction of criminal laws: narcotics includes regulation and prohibition of alcohol",
    "52": "statutory construction of criminal laws: obstruction of justice",
    "53": "statutory construction of criminal laws: perjury (other than as pertains to statutory construction of criminal laws: false statements)",
    "54": "statutory construction of criminal laws: Travel Act, 18 USC 1952",
    "55": "statutory construction of criminal laws: war crimes",
    "56": "statutory construction of criminal laws: sentencing guidelines",
    "57": "statutory construction of criminal laws: miscellaneous",
    "58": "jury trial (right to, as distinct from extra-legal jury influences)",
    "59": "speedy trial",
    "60": "miscellaneous criminal procedure (cf. due process, prisoners' rights, comity: criminal procedure)",
    "61": "voting",
    "62": "Voting Rights Act of 1965, plus amendments",
    "63": "ballot access (of candidates and political parties)",
    "64": "desegregation (other than as pertains to school desegregation, employment discrimination, and affirmative action)",
    "65": "desegregation, schools",
    "66": "employment discrimination: on basis of race, age, religion, illegitimacy, national origin, or working conditions.",
    "67": "affirmative action",
    "68": "slavery or indenture",
    "69": "sit-in demonstrations (protests against racial discrimination in places of public accommodation)",
    "70": "reapportionment: other than plans governed by the Voting Rights Act",
    "71": "debtors' rights",
    "72": "deportation (cf. immigration and naturalization)",
    "73": "employability of aliens (cf. immigration and naturalization)",
    "74": "sex discrimination (excluding sex discrimination in employment)",
    "75": "sex discrimination in employment (cf. sex discrimination)",
    "76": "Indians (other than pertains to state jurisdiction over)",
    "77": "Indians, state jurisdiction over",
    "78": "juveniles (cf. rights of illegitimates)",
    "79": "poverty law, constitutional",
    "80": "poverty law, statutory: welfare benefits, typically under some Social Security Act provision.",
    "81": "illegitimates, rights of (cf. juveniles): typically inheritance and survivor's benefits, and paternity suits",
    "82": "handicapped, rights of: under Rehabilitation, Americans with Disabilities Act, and related statutes",
    "83": "residency requirements: durational, plus discrimination against nonresidents",
    "84": "military: draftee, or person subject to induction",
    "85": "military: active duty",
    "86": "military: veteran",
    "87": "immigration and naturalization: permanent residence",
    "88": "immigration and naturalization: citizenship",
    "89": "immigration and naturalization: loss of citizenship, denaturalization",
    "90": "immigration and naturalization: access to public education",
    "91": "immigration and naturalization: welfare benefits",
    "92": "immigration and naturalization: miscellaneous",
    "93": "indigents: appointment of counsel (cf. right to counsel)",
    "94": "indigents: inadequate representation by counsel (cf. right to counsel)",
    "95": "indigents: payment of fine",
    "96": "indigents: costs or filing fees",
    "97": "indigents: U.S. Supreme Court docketing fee",
    "98": "indigents: transcript",
    "99": "indigents: assistance of psychiatrist",
    "100": "indigents: miscellaneous",
    "101": "liability, civil rights acts (cf. liability, governmental and liability, nongovernmental; cruel and unusual punishment, non-death penalty)",
    "102": "miscellaneous civil rights (cf. comity: civil rights)",
    "103": "First Amendment, miscellaneous (cf. comity: First Amendment)",
    "104": "commercial speech, excluding attorneys",
    "105": "libel, defamation: defamation of public officials and public and private persons",
    "106": "libel, privacy: true and false light invasions of privacy",
    "107": "legislative investigations: concerning internal security only",
    "108": "federal or state internal security legislation: Smith, Internal Security, and related federal statutes",
    "109": "loyalty oath or non-Communist affidavit (other than bar applicants, government employees, political party, or teacher)",
    "110": "loyalty oath: bar applicants (cf. admission to bar, state or federal or U.S. Supreme Court)",
    "111": "loyalty oath: government employees",
    "112": "loyalty oath: political party",
    "113": "loyalty oath: teachers",
    "114": "security risks: denial of benefits or dismissal of employees for reasons other than failure to meet loyalty oath requirements",
    "115": "conscientious objectors (cf. military draftee or military active duty) to military service",
    "116": "campaign spending (cf. governmental corruption):",
    "117": "protest demonstrations (other than as pertains to sit-in demonstrations): demonstrations and other forms of protest based on First Amendment guarantees",
    "118": "free exercise of religion",
    "119": "establishment of religion (other than as pertains to parochiaid:)",
    "120": "parochiaid: government aid to religious schools, or religious requirements in public schools",
    "121": "obscenity, state (cf. comity: privacy): including the regulation of sexually explicit material under the 21st Amendment",
    "122": "obscenity, federal",
    "123": "due process: miscellaneous (cf. loyalty oath), the residual code",
    "124": "due process: hearing or notice (other than as pertains to government employees or prisoners' rights)",
    "125": "due process: hearing, government employees",
    "126": "due process: prisoners' rights and defendants' rights",
    "127": "due process: impartial decision maker",
    "128": "due process: jurisdiction (jurisdiction over non-resident litigants)",
    "129": "due process: takings clause, or other non-constitutional governmental taking of property",
    "130": "privacy (cf. libel, comity: privacy)",
    "131": "abortion: including contraceptives",
    "132": "right to die",
    "133": "Freedom of Information Act and related federal or state statutes or regulations",
    "134": "attorneys' and governmental employees' or officials' fees or compensation or licenses",
    "135": "commercial speech, attorneys (cf. commercial speech)",
    "136": "admission to a state or federal bar, disbarment, and attorney discipline (cf. loyalty oath: bar applicants)",
    "137": "admission to, or disbarment from, Bar of the U.S. Supreme Court",
    "138": "arbitration (in the context of labor-management or employer-employee relations) (cf. arbitration)",
    "139": "union antitrust: legality of anticompetitive union activity",
    "140": "union or closed shop: includes agency shop litigation",
    "141": "Fair Labor Standards Act",
    "142": "Occupational Safety and Health Act",
    "143": "union-union member dispute (except as pertains to union or closed shop)",
    "144": "labor-management disputes: bargaining",
    "145": "labor-management disputes: employee discharge",
    "146": "labor-management disputes: distribution of union literature",
    "147": "labor-management disputes: representative election",
    "148": "labor-management disputes: antistrike injunction",
    "149": "labor-management disputes: jurisdictional dispute",
    "150": "labor-management disputes: right to organize",
    "151": "labor-management disputes: picketing",
    "152": "labor-management disputes: secondary activity",
    "153": "labor-management disputes: no-strike clause",
    "154": "labor-management disputes: union representatives",
    "155": "labor-management disputes: union trust funds (cf. ERISA)",
    "156": "labor-management disputes: working conditions",
    "157": "labor-management disputes: miscellaneous dispute",
    "158": "miscellaneous union",
    "159": "antitrust (except in the context of mergers and union antitrust)",
    "160": "mergers",
    "161": "bankruptcy (except in the context of priority of federal fiscal claims)",
    "162": "sufficiency of evidence: typically in the context of a jury's determination of compensation for injury or death",
    "163": "election of remedies: legal remedies available to injured persons or things",
    "164": "liability, governmental: tort or contract actions by or against government or governmental officials other than defense of criminal actions brought under a civil rights action.",
    "165": "liability, other than as in sufficiency of evidence, election of remedies, punitive damages",
    "166": "liability, punitive damages",
    "167": "Employee Retirement Income Security Act (cf. union trust funds)",
    "168": "state or local government tax",
    "169": "state and territorial land claims",
    "170": "state or local government regulation, especially of business (cf. federal pre-emption of state court jurisdiction, federal pre-emption of state legislation or regulation)",
    "171": "federal or state regulation of securities",
    "172": "natural resources - environmental protection (cf. national supremacy: natural resources, national supremacy: pollution)",
    "173": "corruption, governmental or governmental regulation of other than as in campaign spending",
    "174": "zoning: constitutionality of such ordinances, or restrictions on owners' or lessors' use of real property",
    "175": "arbitration (other than as pertains to labor-management or employeremployee relations (cf. union arbitration)",
    "176": "federal or state consumer protection: typically under the Truth in Lending; Food, Drug and Cosmetic; and Consumer Protection Credit Acts",
    "177": "patents and copyrights: patent",
    "178": "patents and copyrights: copyright",
    "179": "patents and copyrights: trademark",
    "180": "patents and copyrights: patentability of computer processes",
    "181": "federal or state regulation of transportation regulation: railroad",
    "182": "federal and some few state regulations of transportation regulation: boat",
    "183": "federal and some few state regulation of transportation regulation:truck, or motor carrier",
    "184": "federal and some few state regulation of transportation regulation: pipeline (cf. federal public utilities regulation: gas pipeline)",
    "185": "federal and some few state regulation of transportation regulation: airline",
    "186": "federal and some few state regulation of public utilities regulation: electric power",
    "187": "federal and some few state regulation of public utilities regulation: nuclear power",
    "188": "federal and some few state regulation of public utilities regulation: oil producer",
    "189": "federal and some few state regulation of public utilities regulation: gas producer",
    "190": "federal and some few state regulation of public utilities regulation: gas pipeline (cf. federal transportation regulation: pipeline)",
    "191": "federal and some few state regulation of public utilities regulation: radio and television (cf. cable television)",
    "192": "federal and some few state regulation of public utilities regulation: cable television (cf. radio and television)",
    "193": "federal and some few state regulations of public utilities regulation: telephone or telegraph company",
    "194": "miscellaneous economic regulation",
    "195": "comity: civil rights",
    "196": "comity: criminal procedure",
    "197": "comity: First Amendment",
    "198": "comity: habeas corpus",
    "199": "comity: military",
    "200": "comity: obscenity",
    "201": "comity: privacy",
    "202": "comity: miscellaneous",
    "203": "comity primarily removal cases, civil procedure (cf. comity, criminal and First Amendment); deference to foreign judicial tribunals",
    "204": "assessment of costs or damages: as part of a court order",
    "205": "Federal Rules of Civil Procedure including Supreme Court Rules, application of the Federal Rules of Evidence, Federal Rules of Appellate Procedure in civil litigation, Circuit Court Rules, and state rules and admiralty rules",
    "206": "judicial review of administrative agency's or administrative official's actions and procedures",
    "207": "mootness (cf. standing to sue: live dispute)",
    "208": "venue",
    "209": "no merits: writ improvidently granted",
    "210": "no merits: dismissed or affirmed for want of a substantial or properly presented federal question, or a nonsuit",
    "211": "no merits: dismissed or affirmed for want of jurisdiction (cf. judicial administration: Supreme Court jurisdiction or authority on appeal from federal district courts or courts of appeals)",
    "212": "no merits: adequate non-federal grounds for decision",
    "213": "no merits: remand to determine basis of state or federal court decision (cf. judicial administration: state law)",
    "214": "no merits: miscellaneous",
    "215": "standing to sue: adversary parties",
    "216": "standing to sue: direct injury",
    "217": "standing to sue: legal injury",
    "218": "standing to sue: personal injury",
    "219": "standing to sue: justiciable question",
    "220": "standing to sue: live dispute",
    "221": "standing to sue: parens patriae standing",
    "222": "standing to sue: statutory standing",
    "223": "standing to sue: private or implied cause of action",
    "224": "standing to sue: taxpayer's suit",
    "225": "standing to sue: miscellaneous",
    "226": "judicial administration: jurisdiction or authority of federal district courts or territorial courts",
    "227": "judicial administration: jurisdiction or authority of federal courts of appeals",
    "228": "judicial administration: Supreme Court jurisdiction or authority on appeal or writ of error, from federal district courts or courts of appeals (cf. 753)",
    "229": "judicial administration: Supreme Court jurisdiction or authority on appeal or writ of error, from highest state court",
    "230": "judicial administration: jurisdiction or authority of the Court of Claims",
    "231": "judicial administration: Supreme Court's original jurisdiction",
    "232": "judicial administration: review of non-final order",
    "233": "judicial administration: change in state law (cf. no merits: remand to determine basis of state court decision)",
    "234": "judicial administration: federal question (cf. no merits: dismissed for want of a substantial or properly presented federal question)",
    "235": "judicial administration: ancillary or pendent jurisdiction",
    "236": "judicial administration: extraordinary relief (e.g., mandamus, injunction)",
    "237": "judicial administration: certification (cf. objection to reason for denial of certiorari or appeal)",
    "238": "judicial administration: resolution of circuit conflict, or conflict between or among other courts",
    "239": "judicial administration: objection to reason for denial of certiorari or appeal",
    "240": "judicial administration: collateral estoppel or res judicata",
    "241": "judicial administration: interpleader",
    "242": "judicial administration: untimely filing",
    "243": "judicial administration: Act of State doctrine",
    "244": "judicial administration: miscellaneous",
    "245": "Supreme Court's certiorari, writ of error, or appeals jurisdiction",
    "246": "miscellaneous judicial power, especially diversity jurisdiction",
    "247": "federal-state ownership dispute (cf. Submerged Lands Act)",
    "248": "federal pre-emption of state court jurisdiction",
    "249": "federal pre-emption of state legislation or regulation. cf. state regulation of business. rarely involves union activity. Does not involve constitutional interpretation unless the Court says it does.",
    "250": "Submerged Lands Act (cf. federal-state ownership dispute)",
    "251": "national supremacy: commodities",
    "252": "national supremacy: intergovernmental tax immunity",
    "253": "national supremacy: marital and family relationships and property, including obligation of child support",
    "254": "national supremacy: natural resources (cf. natural resources - environmental protection)",
    "255": "national supremacy: pollution, air or water (cf. natural resources environmental protection)",
    "256": "national supremacy: public utilities (cf. federal public utilities regulation)",
    "257": "national supremacy: state tax (cf. state tax)",
    "258": "national supremacy: miscellaneous",
    "259": "miscellaneous federalism",
    "260": "boundary dispute between states",
    "261": "non-real property dispute between states",
    "262": "miscellaneous interstate relations conflict",
    "263": "incorporation of foreign territories",
    "264": "federal taxation, typically under provisions of the Internal Revenue Code",
    "265": "federal taxation of gifts, personal, business, or professional expenses",
    "266": "priority of federal fiscal claims: over those of the states or private entities",
    "267": "miscellaneous federal taxation (cf. national supremacy: state tax)",
    "268": "legislative veto",
    "269": "executive authority vis-a-vis congress or the states",
    "270": "miscellaneous",
    "271": "real property",
    "272": "personal property",
    "273": "contracts",
    "274": "evidence",
    "275": "civil procedure",
    "276": "torts",
    "277": "wills and trusts",
    "278": "commercial transactions"
} 

In [5]:
# Load Supreme Court dataset
sc = textacy.datasets.SupremeCourt()
sc.download()

# Area code classification for 15 labels
# print(sc.issue_area_codes)

mapper = sc.issue_area_codes

# Create a DataFrame from the dataset
cases = [{"id": record.meta["issue"] ,"text": record.text , "actual_class": str(mapper[record.meta['issue_area']]) } for record in sc.records()]

cases_df = pd.DataFrame(cases)

cases_df = cases_df[:10]
# Display the first few rows of the dataset
cases_df.head()
# cases_df = cases_df[:100]

,id,text,actual_class
0,80180,[ Halliburton Oil Well Cementing Co. v. Walker...,Economic Activity
1,10500,"Rehearing Denied Dec. 16, 1946. See . Mr.Claud...",Criminal Procedure
2,80250,"Rehearing Denied Dec. 16, 1946\n See .\n Appea...",Economic Activity
3,20150,"Mr.\nWalter J. Cummings, Jr., of Washington, D...",Civil Rights
4,80060,"Mr.A. Devitt Vaneck, of Washington, D.C., for ...",Economic Activity


In [6]:
# Get words from model's perpective
response = ollama.chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': '''Give me a list of words and phrases which would be present in a case text to classify it into following categories. Do not write any explaination or warnings.
            {legal_topics}

    Format the output in the following way:
    'Category': words, phrase ...
      ''',
  },
])

In [10]:
print(response['message']['content'])

Here is the list of words and phrases that could help classify a text into legal topics:

{legal_topics}: litigation, lawsuit, court, trial, judge, jury, verdict, settlement, arbitration, dispute, contract, agreement, treaty, statute, regulation, ordinance, constitutional, jurisdiction, tort, damages, judgment


In [7]:


def parse_text_to_dict(text):
    result_dict = {}
    lines = text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line or ':' not in line:
            continue  # Ignore headers or tails
        match = re.match(r"^(.*?):\s*(.*)$", line)
        if match:
            category = match.group(1).strip()
            values = [word.strip() for word in match.group(2).split(',')]
            result_dict[category] = values
    
    return result_dict

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)  # Ensure input is a string
    return re.sub(r'[^a-zA-Z0-9 ]', '', text).strip().lower()

def filter_result_dict(result_dict, valid_categories):
    cleaned_valid_categories = {clean_text(str(value)) for value in valid_categories.values()}
    cleaned_result_dict = {clean_text(key): values for key, values in result_dict.items()}
    return {key: values for key, values in cleaned_result_dict.items() if key in cleaned_valid_categories}

# Convert text to dictionary
result = parse_text_to_dict(response['message']['content'])
# for i in result:
#     print(i)
#     print(result[i])

filtered_result = filter_result_dict(result, sc.issue_area_codes)
for i in filtered_result:
    print(i)
    print(filtered_result[i])
    

In [8]:
# Convert dataset records to a DataFrame
records = list(sc.records())
df = pd.DataFrame(records)

# Display the first few rows
print(df.head())

                                                text  \
0  [ Halliburton Oil Well Cementing Co. v. Walker...   
1  Rehearing Denied Dec. 16, 1946. See . Mr.Claud...   
2  Rehearing Denied Dec. 16, 1946\n See .\n Appea...   
3  Mr.\nWalter J. Cummings, Jr., of Washington, D...   
4  Mr.A. Devitt Vaneck, of Washington, D.C., for ...   

                                                meta  
0  {'issue': '80180', 'issue_area': 8, 'n_min_vot...  
1  {'issue': '10500', 'issue_area': 1, 'n_min_vot...  
2  {'issue': '80250', 'issue_area': 8, 'n_min_vot...  
3  {'issue': '20150', 'issue_area': 2, 'n_min_vot...  
4  {'issue': '80060', 'issue_area': 8, 'n_min_vot...  


In [ ]:
# Download necessary resources
nltk.download("stopwords")

# def classify_case_rag(case_text, classification_dict):
#     """
#     Dynamically classify a case using Retrieval-Augmented Generation (RAG) by querying relevant legal topics
#     from all categories in the dictionary.
    
#     Args:
#         case_text (str): The text of the case to be classified.
#         classification_dict (dict): Dictionary mapping classification categories to relevant keywords.
    
#     Returns:
#         str: The best-matching class name for the case.
#     """
    
#     # Load the embedding model
#     model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    
#     # Define the ChromaDB path
#     chroma_db_path = "./chroma_db"
    
#     # Initialize a persistent ChromaDB client
#     chroma_client = chromadb.PersistentClient(path=chroma_db_path)
    
#     # Check if the collection already exists
#     existing_collections = chroma_client.list_collections()
#     if "temp_case_classification" in [col for col in existing_collections]:
#         collection = chroma_client.get_collection("temp_case_classification")
#         print(0)
#     else:
#         collection = chroma_client.create_collection(name="temp_case_classification")

#         # Populate ChromaDB dynamically with classification categories and keywords
#         for class_name, keywords in classification_dict.items():
#             for keyword in keywords:
#                 embedding = model.encode(keyword).tolist()
#                 collection.add(embeddings=[embedding], metadatas=[{"class_name": class_name}], ids=[keyword])
    
#     # Encode the case text for retrieval
#     case_embedding = model.encode(case_text).tolist()
    
#     # Retrieve the most relevant legal categories
#     results = collection.query(query_embeddings=[case_embedding], n_results=3)
    
#     retrieved_classes = [
#         res["class_name"] for res in results["metadatas"][0]
#     ] if results["metadatas"] else []
    
#     if not retrieved_classes:
#         return "No relevant classification found"
    
#     # Prepare relevant context for LLaMA 3
#     retrieved_text = "\n".join(retrieved_classes)

#     prompt = f"""
#     You are an AI assistant trained in Supreme Court case classification. Given the following retrieved categories,
#     classify the new case into the most appropriate issue area.

#     Retrieved Relevant Categories:
#     {retrieved_text}



#     Please reply with the category only and nothing else."""

#     #  Case Text:
#     # {case_text[:1000]}  # Limiting case text for concise processing
    
#     # Query LLaMA 3 using the Ollama instance
#     response = ollama.chat(model='llama3', messages=[{"role": "user", "content": prompt}])
#     # Explicitly delete case_embedding to free memory
#     del case_embedding
#     torch.cuda.empty_cache()  # Clear unused memory from GPU
    
#     return response['message']['content']


import torch
from sentence_transformers import SentenceTransformer

def classify_case_rag(case_text, classification_dict):
    """
    Dynamically classify a case using Retrieval-Augmented Generation (RAG) by querying relevant legal topics.
    
    Args:
        case_text (str): The text of the case to be classified.
        classification_dict (dict): Dictionary mapping classification categories to relevant keywords.
    
    Returns:
        str: The best-matching class name for the case.
    """
    
    # Check GPU availability
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load the embedding model and move it to GPU
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)

    # If multiple GPUs, use DataParallel
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for inference.")
        model = torch.nn.DataParallel(model)
    
    # Function to handle encoding with multi-GPU support
    def encode_text(text):
        if torch.cuda.device_count() > 1:
            return model.module.encode(text, convert_to_tensor=True).to(device)
        else:
            return model.encode(text, convert_to_tensor=True).to(device)
    
    # Define the ChromaDB path
    chroma_db_path = "./chroma_db"
    
    # Initialize a persistent ChromaDB client
    chroma_client = chromadb.PersistentClient(path=chroma_db_path)
    
    # Check if the collection already exists
    existing_collections = chroma_client.list_collections()
    if "temp_case_classification" in [col.name for col in existing_collections]:
        collection = chroma_client.get_collection("temp_case_classification")
    else:
        collection = chroma_client.create_collection(name="temp_case_classification")

        # Populate ChromaDB dynamically with classification categories and keywords
        for class_name, keywords in classification_dict.items():
            for keyword in keywords:
                embedding = encode_text(keyword)
                collection.add(embeddings=[embedding.cpu().tolist()], 
                               metadatas=[{"class_name": class_name}], 
                               ids=[keyword])
    
    # Encode the case text for retrieval (ensure it's on GPU)
    case_embedding = encode_text(case_text)

    # Retrieve the most relevant legal categories
    results = collection.query(query_embeddings=[case_embedding.cpu().tolist()], n_results=3)
    
    retrieved_classes = [
        res["class_name"] for res in results["metadatas"][0]
    ] if results["metadatas"] else []
    
    if not retrieved_classes:
        return "No relevant classification found"
    
    # Prepare relevant context for LLaMA 3
    retrieved_text = "\n".join(retrieved_classes)

    prompt = f"""
    You are an AI assistant trained in Supreme Court case classification. Given the following retrieved categories,
    classify the new case into the most appropriate issue area.

    Retrieved Relevant Categories:
    {retrieved_text}

    Please reply with the category only and nothing else. Ensure its category is from the given list or the most relevant one. Here are the categories:
            - None: Cases that do not fit into any specific category.
            - Criminal Procedure: Cases involving the process of investigating and prosecuting crimes.
            - Civil Rights: Cases about the rights of individuals to receive equal treatment.
            - First Amendment: Cases addressing issues related to freedom of speech, religion, or press.
            - Due Process: Cases focusing on legal safeguards ensuring fair treatment.
            - Privacy: Cases dealing with the right to privacy.
            - Attorneys: Cases about the legal profession and lawyer regulations.
            - Unions: Cases concerning labor unions and collective bargaining.
            - Economic Activity: Cases involving business, trade, or commerce.
            - Judicial Power: Cases addressing the powers of courts and the judiciary.
            - Federalism: Cases about the division of power between state and federal governments.
            - Interstate Relations: Cases about interactions between states.
            - Federal Taxation: Cases involving federal tax laws.
            - Miscellaneous: Cases that do not clearly fit into any of the above categories.
            - Private Action: Cases involving disputes between private individuals or entities.
    """

    # Query LLaMA 3 using the Ollama instance
    response = ollama.chat(model='llama3', messages=[{
                "role": "system",
                "content": '''
                You are an expert in Supreme Court cases and legal classifications. Your task is to analyze the provided text of a Supreme Court case and classify it into one of the following predefined categories:
                
                Instructions:
                1. Carefully analyze the text and identify the primary category it belongs to.
                2. Provide only the category name as the response and nothing else.
                3. If uncertain, select the most relevant category based on the case's primary legal focus.
                4. Response should not exceed more than 2 words.
                ''',
            },{"role": "user", "content": prompt}])
    
    # Free GPU memory
    del case_embedding
    torch.cuda.empty_cache()

    return response['message']['content']


In [ ]:
# %%capture
# Process and classify cases
import gc
classified_cases = []

def process_cases():
    """
    Process all cases synchronously and classify them using the local API.
    """
    for idx, row in cases_df.iterrows():
        print(f"Processing case {idx + 1}/{len(cases_df)}...")

        # Classify the case
        with torch.no_grad():
            class_name = classify_case_rag(row["text"], filtered_result)  # Pass the preprocessed text

        if class_name:
            classified_cases.append({"id": row["id"], "predicted_class_name": class_name})
        else:
            print(f"Failed to classify case {idx + 1}")
        torch.cuda.empty_cache()
        gc.collect()

        # break
    
    print("Processing completed.")

# Run the case processing
process_cases()



In [ ]:
# Save the results to a DataFrame
classified_df = pd.DataFrame(classified_cases)

# filling none for Nan values
classified_df.fillna({'class_name': 'None'}, inplace=True)

# Display the results
classified_df.head()

In [ ]:
merged_df = pd.merge(cases_df,classified_df, left_index=True,right_index=True)  # Use "left", "right", or "outer" if needed
merged_df.head()

In [ ]:
merged_df.to_csv("classified_rag_llama3.csv")

In [ ]:
from sklearn.metrics import accuracy_score

# Assuming your DataFrame is named df
accuracy = accuracy_score(merged_df["actual_class"], merged_df["predicted_class_name"])
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(merged_df["actual_class"], merged_df["predicted_class_name"], labels=merged_df["actual_class"].unique())

# Create a heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=merged_df["actual_class"].unique(), yticklabels=merged_df["actual_class"].unique())
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import time

# Assuming merged_df contains your actual and predicted classes
# Create a new directory based on the current timestamp
timestamp = time.strftime("%Y%m%d-%H%M%S")
folder_name = f"run_{timestamp}"
os.makedirs(folder_name, exist_ok=True)

# Save the CSV file
csv_file_path = os.path.join(folder_name, "classified_rag_llama3.csv")
merged_df.to_csv(csv_file_path, index=False)

# Calculate accuracy
accuracy = accuracy_score(merged_df["actual_class"], merged_df["predicted_class_name"])
print(f"Accuracy: {accuracy:.2%}")

# Save accuracy to a text file
accuracy_file_path = os.path.join(folder_name, "accuracy.txt")
with open(accuracy_file_path, "w") as f:
    f.write(f"Accuracy: {accuracy:.2%}\n")

# Compute confusion matrix
cm = confusion_matrix(merged_df["actual_class"], merged_df["predicted_class_name"], labels=merged_df["actual_class"].unique())

# Create a heatmap of the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=merged_df["actual_class"].unique(), yticklabels=merged_df["actual_class"].unique())
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Confusion Matrix")

# Save the confusion matrix plot as a PNG file
confusion_matrix_png_path = os.path.join(folder_name, "confusion_matrix.png")
plt.savefig(confusion_matrix_png_path)

# Close the plot to free up memory
plt.close()

print(f"Files saved in folder: {folder_name}")
